
## 3. Preprocesamiento implementado en código

Se implementa el pipeline de preprocesamiento completo, dejando los datos listos para modelado.

In [4]:
from pathlib import Path
import anndata as ad
import numpy as np

file_path = Path("..") / "data" / "raw" / "adata_final.h5ad"
adata = ad.read_h5ad(file_path)

In [5]:
# --- 3.1 Extracción de features y variable objetivo ---
X = adata.obsm['X_pca']
y_raw = adata.obs['lineage'].astype(str).values

print('Shape de features (X_pca):', X.shape)
print('Shape de target (lineage):', y_raw.shape)
print('Clases únicas:', np.unique(y_raw))

Shape de features (X_pca): (328170, 50)
Shape de target (lineage): (328170,)
Clases únicas: ['B&plasma' 'MNP' 'NK' 'T' 'mast' 'pDC']


In [6]:
# --- 3.2 Encoding manual de la variable objetivo ---

mapping = {
    "T": 0,
    "MNP": 1,
    "B&plasma": 2,
    "NK": 3,
    "mast": 4,
    "pDC": 5
}

# Aplicar mapping
y = adata.obs["lineage"].map(mapping).values

print("Mapping utilizado:")
for k, v in mapping.items():
    print(f"  {v} -> {k}")

Mapping utilizado:
  0 -> T
  1 -> MNP
  2 -> B&plasma
  3 -> NK
  4 -> mast
  5 -> pDC


In [7]:
from sklearn.model_selection import train_test_split
import pandas as pd

# --- 3.3 Train/Test split estratificado ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Tamaño del conjunto de entrenamiento:', X_train.shape)
print('Tamaño del conjunto de prueba:       ', X_test.shape)

#Crear mapping inverso
inv_mapping = {v: k for k, v in mapping.items()}

print('\nDistribución de clases en train:')
train_dist = pd.Series(y_train).map(inv_mapping).value_counts(normalize=True).round(4)
print(train_dist.to_string())

print('\nDistribución de clases en test:')
test_dist = pd.Series(y_test).map(inv_mapping).value_counts(normalize=True).round(4)
print(test_dist.to_string())

Tamaño del conjunto de entrenamiento: (262536, 50)
Tamaño del conjunto de prueba:        (65634, 50)

Distribución de clases en train:
T           0.4804
MNP         0.3250
B&plasma    0.1262
NK          0.0349
mast        0.0277
pDC         0.0058

Distribución de clases en test:
T           0.4804
MNP         0.3250
B&plasma    0.1262
NK          0.0349
mast        0.0277
pDC         0.0058


In [8]:
from pathlib import Path

output_path = Path("..") / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

np.save(output_path / "X_train.npy", X_train)
np.save(output_path / "X_test.npy", X_test)
np.save(output_path / "y_train.npy", y_train)
np.save(output_path / "y_test.npy", y_test)

In [9]:
import joblib

joblib.dump(mapping, output_path / "label_mapping.pkl")

['..\\data\\processed\\label_mapping.pkl']

## **Justificación técnica de las decisiones de preprocesamiento**

| Decisión | Justificación |
|---|---|
| Usar `X_pca` (50 componentes) como features | La matriz original (328K × 2000) es altamente dispersa (~8.2% densidad) y de alta dimensionalidad. El uso de PCA reduce el número de variables a 50 componentes, capturando ~49% de la varianza, eliminando ruido y haciendo el modelado más eficiente y escalable |
| No aplicar normalización adicional | El PCA fue calculado previamente sobre datos normalizados (`log1p`), por lo que las componentes ya se encuentran en una escala adecuada y comparable para modelos de machine learning |
| Encoding manual de `lineage` | La variable objetivo es categórica nominal (sin orden). Se definió un mapping manual para asegurar control, interpretabilidad y reproducibilidad del pipeline, evitando ambigüedades del encoding automático |
| Uso de valores enteros como target | Los modelos supervisados de clasificación (ej. Random Forest, XGBoost) requieren variables objetivo numéricas. Los valores asignados funcionan como identificadores, no representan jerarquía entre clases |
| Split estratificado (80/20) | Se mantiene la proporción original de clases en entrenamiento y prueba, lo cual es crítico debido al desbalance presente en `lineage`. El conjunto de test (~65K células) es suficientemente grande para una evaluación robusta |
| random_state=42 | Permite reproducibilidad del experimento, asegurando que los resultados sean consistentes entre ejecuciones |
| Uso exclusivo de `lineage` como target | Se selecciona `lineage` como variable objetivo por su relevancia biológica (tipo celular general) y porque representa un problema de clasificación supervisada bien definido |



---
## 4. Data Quality Report consolidado

In [11]:
nnz      = adata.X.nnz
total    = adata.n_obs * adata.n_vars
densidad = nnz / total
sparsity = 1 - densidad

nulls_obs  = adata.obs.isnull().sum().sum()
nulls_var  = adata.var.isnull().sum().sum()
n_dup_obs  = adata.obs.duplicated().sum()

# Distribución de clases
lineage_counts = adata.obs["lineage"].value_counts()

# --- Reporte ---
print("=" * 60)
print(" DATA QUALITY REPORT")
print("=" * 60)

# 1. Dimensiones
print("\n 1. Dimensiones del dataset")
print(f"Células (obs): {adata.n_obs:,}")
print(f"Genes (vars):  {adata.n_vars:,}")
print(f"Shape:         {adata.shape}")

# 2. Completitud
print("\n 2. Completitud de los datos")
print(f"Valores nulos en obs: {nulls_obs}")
print(f"Valores nulos en var: {nulls_var}")
print("Completitud: 100%")

# 3. Matriz X
print("\n 3. Estructura de la matriz X")
print(f"Valores no nulos (nnz): {nnz:,}")
print(f"Densidad: {densidad:.4f}")
print(f"Sparsity: {sparsity:.4f}")
print(f"Tipo de matriz: {type(adata.X).__name__}")

# 4. Duplicados
print("\n 4. Duplicados")
print(f"Filas duplicadas en obs: {n_dup_obs}")

# 5. Distribución de clases
print("\n 5. Distribución de clases (lineage)")
total_cells = lineage_counts.sum()

for cls, cnt in lineage_counts.items():
    pct = (cnt / total_cells) * 100
    print(f"{cls:<12} {cnt:>10,} ({pct:.2f}%)")

# 6. Estructuras adicionales
print("\n 6. Estructuras adicionales")
print("Embeddings:", list(adata.obsm.keys()))
print("Matrices de relaciones:", list(adata.obsp.keys()))
print("Info genes (varm):", list(adata.varm.keys()))

print("\n" + "=" * 60)

 DATA QUALITY REPORT

 1. Dimensiones del dataset
Células (obs): 328,170
Genes (vars):  2,000
Shape:         (328170, 2000)

 2. Completitud de los datos
Valores nulos en obs: 0
Valores nulos en var: 0
Completitud: 100%

 3. Estructura de la matriz X
Valores no nulos (nnz): 53,831,003
Densidad: 0.0820
Sparsity: 0.9180
Tipo de matriz: csc_matrix

 4. Duplicados
Filas duplicadas en obs: 324130

 5. Distribución de clases (lineage)
T               157,641 (48.04%)
MNP             106,662 (32.50%)
B&plasma         41,422 (12.62%)
NK               11,456 (3.49%)
mast              9,082 (2.77%)
pDC               1,907 (0.58%)

 6. Estructuras adicionales
Embeddings: ['X_pca', 'X_umap']
Matrices de relaciones: ['connectivities', 'distances']
Info genes (varm): ['PCs']



In [12]:
riesgos = [
    ('Desbalance de clases',
     'Clases minoritarias (pDC, mast) vs. mayoritarias (T, MNP). '
     'Riesgo: el modelo puede ignorar clases minoritarias. '
     "Mitigación: usar class_weight='balanced', SMOTE, o métricas macro."),
    ('Alta sparsity (~91.8%)',
     'La mayoría de valores en la matriz de expresión son cero (dropouts). '
     'Riesgo: modelos sensibles a escala pueden verse afectados. '
     'Mitigación: usar PCA (ya aplicado).'),
    ('Fuente del dataset no publicada',
     'El dataset fue proporcionado por un contacto académico de la UTEM. '
     'No tiene DOI ni URL pública. '
     'Riesgo: no se puede reproducir la generación del archivo externamente. '
     'Mitigación: el archivo .h5ad contiene el preprocesamiento documentado internamente.'),
    ('PCA captura solo ~49% de varianza',
     'Con 50 componentes se retiene ~mitad de la varianza original. '
     'Riesgo: pérdida de información biológica. '
     'Mitigación: evaluar si aumentar componentes mejora el modelo.'),
    ('Sesgo de batch',
     'El dataset proviene de 91 muestras (sample_ID). '
     'Riesgo: efectos de batch pueden introducir variabilidad técnica. '
     'Mitigación: HVG seleccionados por batch (highly_variable_nbatches).'),
]

print('RIESGOS Y LIMITACIONES DEL DATASET')
print('=' * 60)
for i, (riesgo, descripcion) in enumerate(riesgos, 1):
    print(f'\n{i}. {riesgo}')
    print(f'   {descripcion}')
print('\n' + '=' * 60)

RIESGOS Y LIMITACIONES DEL DATASET

1. Desbalance de clases
   Clases minoritarias (pDC, mast) vs. mayoritarias (T, MNP). Riesgo: el modelo puede ignorar clases minoritarias. Mitigación: usar class_weight='balanced', SMOTE, o métricas macro.

2. Alta sparsity (~91.8%)
   La mayoría de valores en la matriz de expresión son cero (dropouts). Riesgo: modelos sensibles a escala pueden verse afectados. Mitigación: usar PCA (ya aplicado).

3. Fuente del dataset no publicada
   El dataset fue proporcionado por un contacto académico de la UTEM. No tiene DOI ni URL pública. Riesgo: no se puede reproducir la generación del archivo externamente. Mitigación: el archivo .h5ad contiene el preprocesamiento documentado internamente.

4. PCA captura solo ~49% de varianza
   Con 50 componentes se retiene ~mitad de la varianza original. Riesgo: pérdida de información biológica. Mitigación: evaluar si aumentar componentes mejora el modelo.

5. Sesgo de batch
   El dataset proviene de 91 muestras (sample_ID